In [3]:
!pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# -------------------------------
# Basic libraries
# -------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# Preprocessing
# -------------------------------
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# -------------------------------
# Deep Learning (LSTM)
# -------------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
file_path = r"C:\Users\kanis\OneDrive\Desktop\AircraftProject\outputs\test_anomaly_RUL_results.csv"

df = pd.read_csv(file_path)

In [6]:
print(df.head())
print(df.shape)
print(df.columns)

   unit_number  time_in_cycles  sensor_measurement_2  sensor_measurement_3  \
0            1               1                641.82               1589.70   
1            1               2                642.15               1591.82   
2            1               3                642.35               1587.99   
3            1               4                642.35               1582.79   
4            1               5                642.37               1582.85   

   sensor_measurement_4  sensor_measurement_7  sensor_measurement_9  \
0               1400.60                554.36               9046.19   
1               1403.14                553.75               9044.07   
2               1404.20                554.26               9052.94   
3               1401.87                554.45               9049.48   
4               1406.22                554.00               9055.15   

   sensor_measurement_11  sensor_measurement_12  sensor_measurement_17  \
0                  47.47      

In [7]:
df = df.drop(columns=['max_cycle'])


In [8]:
df['anomaly'] = df['anomaly'].map({-1: 1, 1: 0})

In [9]:
feature_cols = [
    col for col in df.columns
    if col not in ['unit_number', 'time_in_cycles', 'RUL']
]

In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

In [11]:
df.to_csv("FD001_after_anomaly_scaled.csv", index=False)

In [12]:
#LSTM MODEL TRAINING
import numpy as np

def create_sequences(df, feature_cols, window_size=30):
    X, y = [], []

    for uid in df['unit_number'].unique():
        engine_data = df[df['unit_number'] == uid]

        for i in range(len(engine_data) - window_size):
            X.append(
                engine_data.iloc[i:i+window_size][feature_cols].values
            )
            y.append(
                engine_data.iloc[i+window_size]['RUL']
            )

    return np.array(X), np.array(y)

In [13]:
WINDOW_SIZE = 30

X_train, y_train = create_sequences(
    df,
    feature_cols,
    WINDOW_SIZE
)

print(X_train.shape)
print(y_train.shape)

(17631, 30, 11)
(17631,)


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),

    LSTM(32),
    Dropout(0.2),

    Dense(1)   # RUL
])

model.compile(
    optimizer='adam',
    loss='mse', 
    metrics = ['accuracy']
)

model.summary()

C:\Users\kanis\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 30, 64)              │          19,456 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 30, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 31,905 (124.63 KB)

 Trainable params: 31,905 (124.63 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.0056 - loss: 10753.1025
Epoch 2/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 9260.7256
Epoch 3/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 8123.8848
Epoch 4/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 7184.6270
Epoch 5/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 6413.9678
Epoch 6/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 5783.3687
Epoch 7/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 5274.5688
Epoch 8/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 4866.6494
Epoch 9/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 9s 34ms/step - accuracy: 0.0057 - loss: 4572.6133
Epoch 10/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.0057 - loss: 4340.0464
Epoch 11/50
276/276 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - accuracy: 0.0057 - loss: 4169.1289
Epoc